In [ ]:
import os
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ===========================================================================
# STEP 1: LINK AND EXPLORE THE DATASET IN FOLDER '221666051'
# ===========================================================================
base_dir = os.path.dirname(os.path.abspath(__file__)) if "__file__" in locals() else os.getcwd()
dataset_dir = os.path.join(base_dir, "221666051")

# Handle nested folder structure if present (221666051/221666051)
if os.path.exists(os.path.join(dataset_dir, "221666051")):
    dataset_dir = os.path.join(dataset_dir, "221666051")

print(f"🔗 Dataset Linked Path: {dataset_dir}")

# List and inspect dataset files
dataset_files = os.listdir(dataset_dir)
print(f"\nFiles found ({len(dataset_files)} items):")
for filename in dataset_files:
    f_path = os.path.join(dataset_dir, filename)
    size_mb = os.path.getsize(f_path) / (1024 * 1024)
    print(f" - {filename:<48} ({size_mb:>6.2f} MB)")

# ===========================================================================
# STEP 2: PARSE METADATA (BAND_META.txt)
# ===========================================================================
meta_file = os.path.join(dataset_dir, "BAND_META.txt")
meta = {}
if os.path.exists(meta_file):
    with open(meta_file, "r") as f:
        for line in f:
            if "=" in line:
                k, v = line.split("=", 1)
                meta[k.strip()] = v.strip()

product_id = meta.get("ProductID", "221666051")
sat_id = meta.get("SatID", "IRS-R2A")
sensor = meta.get("Sensor", "L3")
date_of_pass = meta.get("DateOfPass", "18-APR-2021")
resolution = float(meta.get("OutputResolutionAlong", 24.0))
num_scans = int(meta.get("NoScans", 7113))
num_pixels = int(meta.get("NoPixels", 7744))

# Mapping bands with their corresponding physical properties
band_dict = {
    "BAND2": ("Band 2 (Green)", float(meta.get("B2Temp", 17.89)), float(meta.get("B2_Lmin", 0.0)), float(meta.get("B2_Lmax", 52.0))),
    "BAND3": ("Band 3 (Red)",   float(meta.get("B3Temp", 18.20)), float(meta.get("B3_Lmin", 0.0)), float(meta.get("B3_Lmax", 47.0))),
    "BAND4": ("Band 4 (NIR)",   float(meta.get("B4Temp", 17.89)), float(meta.get("B4_Lmin", 0.0)), float(meta.get("B4_Lmax", 31.5))),
    "BAND5": ("Band 5 (SWIR)",  float(meta.get("B5Temp", 23.75)), float(meta.get("B5_Lmin", 0.0)), float(meta.get("B5_Lmax", 7.5))),
}

# ===========================================================================
# STEP 3: INITIALIZE SQLITE DATABASE & CREATE TABLES
# ===========================================================================
db_path = os.path.join(base_dir, "satellite_data.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Table 1: Satellite Product Metadata
cursor.execute("""
CREATE TABLE IF NOT EXISTS satellite_product (
    product_id TEXT PRIMARY KEY,
    satellite_name TEXT,
    sensor_name TEXT,
    date_of_pass TEXT,
    resolution_meters REAL,
    num_scans INTEGER,
    num_pixels INTEGER
)
""")

# Table 2: Band Catalog linked with actual raster files
cursor.execute("""
CREATE TABLE IF NOT EXISTS band_catalog (
    band_id TEXT PRIMARY KEY,
    band_name TEXT,
    file_name TEXT,
    file_path TEXT,
    file_size_mb REAL,
    temperature_celsius REAL,
    radiance_lmin REAL,
    radiance_lmax REAL,
    product_id TEXT,
    FOREIGN KEY (product_id) REFERENCES satellite_product(product_id)
)
""")
conn.commit()

# ===========================================================================
# STEP 4: INSERT RECORDS INTO DATABASE (CRUD: Create)
# ===========================================================================
cursor.execute("""
INSERT OR REPLACE INTO satellite_product 
(product_id, satellite_name, sensor_name, date_of_pass, resolution_meters, num_scans, num_pixels)
VALUES (?, ?, ?, ?, ?, ?, ?)
""", (product_id, sat_id, sensor, date_of_pass, resolution, num_scans, num_pixels))

for band_key, (name, temp, lmin, lmax) in band_dict.items():
    tif_name = f"{band_key}.tif"
    tif_path = os.path.join(dataset_dir, tif_name)
    file_size_mb = os.path.getsize(tif_path) / (1024 * 1024) if os.path.exists(tif_path) else 0.0
    
    cursor.execute("""
    INSERT OR REPLACE INTO band_catalog
    (band_id, band_name, file_name, file_path, file_size_mb, temperature_celsius, radiance_lmin, radiance_lmax, product_id)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (band_key, name, tif_name, tif_path, round(file_size_mb, 2), temp, lmin, lmax, product_id))

conn.commit()
print("\n✅ Data inserted into SQLite tables successfully!")

# ===========================================================================
# STEP 5: SQL QUERIES & AGGREGATIONS (CRUD: Read)
# ===========================================================================
print("\n--- SQL Query: All Band Records ---")
cursor.execute("SELECT band_id, band_name, file_size_mb, temperature_celsius, radiance_lmax FROM band_catalog")
for row in cursor.fetchall():
    print(f"ID: {row[0]:<6} | {row[1]:<15} | Size: {row[2]:>6.2f} MB | Temp: {row[3]:>5.2f} deg C | Lmax: {row[4]}")

print("\n--- SQL Query with Filter (WHERE temperature_celsius > 18.0) ---")
cursor.execute("SELECT band_name, temperature_celsius FROM band_catalog WHERE temperature_celsius > 18.0")
for name, temp in cursor.fetchall():
    print(f" • {name}: {temp} deg C")

print("\n--- SQL Aggregations (COUNT, AVG, MIN, MAX, SUM) ---")
cursor.execute("""
SELECT 
    COUNT(*),
    AVG(temperature_celsius),
    MIN(temperature_celsius),
    MAX(temperature_celsius),
    SUM(file_size_mb)
FROM band_catalog
""")
total_bands, avg_temp, min_temp, max_temp, total_size = cursor.fetchone()
print(f"Total Bands            : {total_bands}")
print(f"Average Temperature    : {avg_temp:.2f} deg C")
print(f"Min / Max Temperature  : {min_temp:.2f} deg C / {max_temp:.2f} deg C")
print(f"Total Band Data Size   : {total_size:.2f} MB")

# ===========================================================================
# STEP 6: SQL UPDATE OPERATION (CRUD: Update)
# ===========================================================================
cursor.execute("""
UPDATE band_catalog
SET temperature_celsius = 18.04
WHERE band_id = 'BAND2'
""")
conn.commit()

cursor.execute("SELECT band_id, temperature_celsius FROM band_catalog WHERE band_id = 'BAND2'")
print(f"\n✅ Updated BAND2 calibration: {cursor.fetchone()}")

# ===========================================================================
# STEP 7: PANDAS & NUMPY DATA ANALYSIS (Learned in Day 1 & Day 2)
# ===========================================================================
# Reading SQLite directly into a Pandas DataFrame
df = pd.read_sql_query("SELECT * FROM band_catalog", conn)

print("\n--- Pandas DataFrame from Database ---")
print(df[["band_id", "band_name", "file_size_mb", "temperature_celsius", "radiance_lmax"]])

print("\n--- Missing Values Check (.isnull().sum()) ---")
print(df.isnull().sum())

print("\n--- Descriptive Summary (.describe()) ---")
print(df[["temperature_celsius", "radiance_lmax", "file_size_mb"]].describe())

# NumPy array statistics
temp_arr = df["temperature_celsius"].to_numpy()
print(f"\nNumPy Mean Temperature  : {np.mean(temp_arr):.2f}")
print(f"NumPy Median Temperature: {np.median(temp_arr):.2f}")
print(f"NumPy Std Deviation     : {np.std(temp_arr):.2f}")

# Adding a calculated column: Dynamic Radiance Range (Lmax - Lmin)
df["radiance_range"] = df["radiance_lmax"] - df["radiance_lmin"]
print("\n--- Radiance Range (Lmax - Lmin) ---")
print(df[["band_name", "radiance_range"]])

# ===========================================================================
# STEP 8: DATA VISUALIZATION WITH MATPLOTLIB (Learned in Day 1 & Day 2)
# ===========================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

# Subplot 1: Band Temperatures (Bar Chart)
colors = ['#2ca02c', '#d62728', '#9467bd', '#ff7f0e']
ax1.bar(df["band_name"], df["temperature_celsius"], color=colors, edgecolor='black', width=0.5)
ax1.set_title("Band Temperatures (deg C)", fontweight='bold')
ax1.set_ylabel("Temperature (deg C)")
ax1.set_ylim(0, 30)
ax1.grid(axis='y', linestyle='--', alpha=0.7)
for i, v in enumerate(df["temperature_celsius"]):
    ax1.text(i, v + 0.6, f"{v}", ha='center', fontweight='bold')

# Subplot 2: Maximum Spectral Radiance (Line Plot)
ax2.plot(df["band_name"], df["radiance_lmax"], marker='o', color='#1f77b4', lw=2.5, markersize=8)
ax2.set_title("Max Spectral Radiance (Lmax)", fontweight='bold')
ax2.set_ylabel("Radiance (W/m²/sr/μm)")
ax2.grid(True, linestyle='--', alpha=0.7)
for i, v in enumerate(df["radiance_lmax"]):
    ax2.text(i, v + 1.5, f"{v}", ha='center')

plt.tight_layout()
plt.show()

# Close connection
conn.close()
print("\n🔒 Database connection closed cleanly.")
